# Phase 2a-2. DEG analysis & paper marker overlap validation

이 notbook의 목적은 TAM subtype annoation 이후 각 subtype을 특징짓는 DEG(Differentially Expressed Genes)를 찾고 논문에서 제시한 maker gene과 overlap되는지 검증하는 것이다.

핵심 질문:
- C1QC+ TAM과 SPP1+ TAM은 실제로 서로 다른 transcriptional signiture를 가지는가?
- 논문 marker gene이 GSE127465 폐암 데이터에서도 DEG 상위권에 재현되는가?

## 0. 왜 annoation 이후 DEG 분석을 다시 하는가?

Annoation은 marker gene 기반으로 cluster에 이름을 붙이는 interpretation 단계다.

반면 DEG 분석은 각 cluster/subtype을 특징짓는 gene이 실제 데이터에서 통계적으로 유의미하게 높게 발현되는지 확인하는 단계이다.

즉:
- annotatio: known marker를 보고 이름을 붙임
- DEG: 이 그룹을 가장 잘 설명하는 gene signature를 데이터에서 다시 찾음

따라서 DEG는 annation의 근거를 강화하고 논문 marker와의 재현성을 검증하는 역할을 한다

In [ ]:

import scanpy as sc
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
from utils.paths import *

mac = sc.read_h5ad(MAC_SUB_H5AD)

print(mac)
print(mac.obs['tam_subtype'].value_counts())
print('raw exists:', mac.raw is not None)
print('raw shape:', None if mac.raw is None else mac.raw.shape)

## 1. DEG 실행

`rank_gene_groups`는 group별로 해당 group에서 상대적으로 더 높게 발현되는 gene을 ranking한다

여기서는 `tam_subtype`을 기준으로 C1QC+ TAM, SPP1+ TAM, Unknown을 비교한다.

`method='wilcoxon'`을 사용하는 이유:
- scRNA-seq 데이터는 0이 많고(sparse) 분포가 정규분포를 따르지 않는 경우가 많다
- Wilcoxon rank-sum test는 rank 기반 비교라 이런 분포에 비교적 robust하다

`use_raw=True`를 사용하는 이유:
- HVG filltering 이후 `adata.X`에는 일부 gene만 남아 있을 수 있다
- marker/DEG 해석에는 raw normalized matrix 기준 gene expression을 쓰는 것이 안전하다

In [ ]:
sc.tl.rank_genes_groups(
    mac,
    groupby='tam_subtype',
    method='wilcoxon',
    key_added='deg_tam',
    use_raw=True
)

print('DEG result keys:', mac.uns['deg_tam'].keys())
sc.pl.rank_genes_groups(mac, n_genes=20, key='deg_tam', sharey=False)

## 2. DEG 결과 table로 확인

plot만 보면 전체 값을 검토하기 어렵기 때문에 각 subtype의 상위 DEG를 dataframe으로 추출한다.

확인 항목:
- names: DEG gene name
- scores: 해당 group에서 얼마나 특이적인지 나타내는 scroe
- logfoldchanges: group 간 발현 차이 크기
- pvals_adj: multiple testing correction 이후 유익성

In [ ]:
deg_tables = {}
for group in mac.obs['tam_subtype'].cat.categories if hasattr(mac.obs['tam_subtype'], 'cat') else sorted(mac.obs['tam_subtype'].unique()):
    df = sc.get.rank_genes_groups_df(mac, group=group, key='deg_tam')
    deg_tables[group] = df
    print(f"===== {group} top 10 DEG =====")
    display(df[['names', 'scores', 'logfoldchanges', 'pvals_adj']].head(10))

## 3. 논문 marker gene과 내 DEG top 50 overlap 검증

논문 marker gene이 내 데이터의 DEG 상위권에 재현되는지 확인한다.

이 검증의 의미:
- 단순히 UMAP에서 비슷해 보이는 것이 아니라 논문에서 제시한 TAM subtype signature가 내 데이터에서도 통계적으로 상위 DEG로 재현되는지 확인
- 특정 dataset artifact가 아니라 동일 biological program이 재현될 가능성을 높임

단, overlap 5/5는 subtype signature 재현의 강한 근거이지 예후 인과성을 직접 증명한 것은 아니다. 예후 검증에는  survival, treatment response, spatial/clinical data가 추가로 요구된다.

In [ ]:
paper_markers = {
    'C1QC+ TAM': ['C1QA', 'C1QB', 'C1QC', 'APOE', 'FOLR2'],
    'SPP1+ TAM': ['SPP1', 'GPNMB', 'CTSD', 'MRC1', 'CD63'],
}

overlap_rows = []
for group, markers in paper_markers.items():
    my_genes = sc.get.rank_genes_groups_df(mac, group=group, key='deg_tam')['names'].head(50).tolist()
    overlap = sorted(set(my_genes) & set(markers))
    overlap_rows.append({
        'subtype': group,
        'paper_markers': ', '.join(markers),
        'overlap_genes': ', '.join(overlap),
        'overlap_count': f'{len(overlap)}/{len(markers)}',
        'overlap_ratio': len(overlap) / len(markers),
    })

overlap_df = pd.DataFrame(overlap_rows)
display(overlap_df)


## 4. Marker panel dotplot

논문 marker panel을 subtype 별로 시각화하여 C1QC+ TAM과 SPP1+ TAM이 서로 다른 gene program을 가지는지 확인한다.

Dotplot 해석:
- 점 크기: 해당 group에서 그 gene을 발현하는 cell fraction
- 색 진하기: 해당 group 내 평균 발현량
- standard_scale='var': gene별 상대적 발현 패턴을 비교하기 쉽게 scaling

In [ ]:
sc.pl.dotplot(
    mac,
    var_names=['C1QA', 'C1QB', 'C1QC', 'APOE', 'FOLR2',
               'SPP1', 'GPNMB', 'CTSD', 'MRC1', 'CD63'],
    groupby='tam_subtype',
    standard_scale='var'
)

## 5. Biology interpretation note

C1QC+ TAM:
- C1QA/C1QB/C1QC, APOE, FOLR2가 함께 높게 나타난다
- complement, phagocytosis, tissue homeostasis, resident macrophage program과 연결 가능

SPP1+ TAM:
- SPP1, GPNMB, CTSD, MRC1, CD63가 함께 높게 나타난다
- ECM remodeling, invasion, lysosomal/protease activity, immunosuppressive TAM signature와 연결 가능

즉 둘 다 macrophage이지만, 같은 macrophage 안에서도 functional state가 다르다는 점이 핵심이다

In [ ]:
mac.write(MAC_PH2_H5AD)
print(f'저장 완료: {MAC_PH2_H5AD}')